<a href="https://colab.research.google.com/github/germanoisabela/ai-assistant/blob/main/Ia_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU pinecone pinecone-client langchain-pinecone langchain-groq langchain-text-splitters sentence-transformers pypdf gradio langchain



In [10]:
import os
from google.colab import userdata
!pip install langchain-community
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

In [11]:
print("Ambiente configurado e documento pronto.")

Ambiente configurado e documento pronto.


In [12]:
loader = PyPDFLoader("manual_ia.pdf")
data=loader.load()

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
chunks = splitter.split_documents(data)
print(f"Fragmentos gerados: {len(chunks)}")

Fragmentos gerados: 183


In [27]:
from langchain_core.tools import retriever
from langchain_core import embeddings
import os
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_classic.chains import RetrievalQA

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

embeddings = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2")

index_name = "laboratorio-ia"
vectorstore = PineconeVectorStore.from_documents(
    chunks,
    embeddings,
    index_name=index_name
)

template = """Você é um auditor técnico. Analise o contexto abaixo para responder à pergunta.

Contexto: {context}

Diretriz: Responda com base no contexto. Se a informação estiver implícita, explique.
Apenas se o assunto for totalmente diferente do contexto, diga que não encontrou evidências.

Pergunta: {question}
Resposta:"""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant")
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
    chain_type_kwargs={"prompt": prompt}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import gradio as gr

def portal_ia(pergunta, historial):
  resultado = rag_chain.invoke(pergunta)
  return resultado["result"]

with gr.Blocks() as demo:
  gr.Markdown("Corporate AI Knowledge Base")
  gr.Markdown("Consulte documentos oficiais de forme segura e sem alucinações.")

  chat = gr.ChatInterface(fn=portal_ia)

demo.launch(
    share=True,
    debug=True,
    theme=gr.themes.Monochrome()
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://887b518576e5858df6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
